In [1]:
# Cell 1: Install required packages (run once)

%pip install -q pypdf chromadb ollama tqdm

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Cell 2: Imports and configuration

import os

from pathlib import Path

from typing import List, Dict



# Disable Chroma telemetry to avoid noisy telemetry-client errors

os.environ["ANONYMIZED_TELEMETRY"] = "False"



import chromadb

from chromadb.config import Settings

import ollama

from pypdf import PdfReader

from tqdm import tqdm



# Extra guard for chromadb/posthog version mismatch in some environments

try:

    import posthog

    posthog.capture = lambda *args, **kwargs: None

except Exception:

    pass



# Ollama models root from your D: drive setup

OLLAMA_MODELS_DIR = Path(r"D:\\Ollama\\models")

os.environ["OLLAMA_MODELS"] = str(OLLAMA_MODELS_DIR)



# Ollama config

OLLAMA_HOST = "http://localhost:11434"

CHAT_MODEL = "mistral:7b"

EMBED_MODEL = "nomic-embed-text:v1.5"



# Exact manifest paths from your models folder

CHAT_MODEL_MANIFEST = OLLAMA_MODELS_DIR / "manifests" / "registry.ollama.ai" / "library" / "mistral" / "7b"

EMBED_MODEL_MANIFEST = OLLAMA_MODELS_DIR / "manifests" / "registry.ollama.ai" / "library" / "nomic-embed-text" / "v1.5"



# Data and DB paths

PDF_DIR = Path(r"D:\\MyProjects\\AI Project\\Rag_local_Ollama\\pdf")

CHROMA_DIR = Path(r"D:\\MyProjects\\AI Project\\Rag_local_Ollama")

COLLECTION_NAME = "pdf_rag_collection"



# Chunking config (character-based)

CHUNK_SIZE = 1000

CHUNK_OVERLAP = 150



print("Configured.")

print(f"OLLAMA_MODELS_DIR: {OLLAMA_MODELS_DIR}")

print(f"Chat model: {CHAT_MODEL}")

print(f"Embedding model: {EMBED_MODEL}")

print(f"Chat manifest exists: {CHAT_MODEL_MANIFEST.exists()} -> {CHAT_MODEL_MANIFEST}")

print(f"Embed manifest exists: {EMBED_MODEL_MANIFEST.exists()} -> {EMBED_MODEL_MANIFEST}")

print(f"PDF_DIR: {PDF_DIR.resolve()}")

print(f"CHROMA_DIR: {CHROMA_DIR.resolve()}")

Configured.
OLLAMA_MODELS_DIR: D:\Ollama\models
Chat model: mistral:7b
Embedding model: nomic-embed-text:v1.5
Chat manifest exists: True -> D:\Ollama\models\manifests\registry.ollama.ai\library\mistral\7b
Embed manifest exists: True -> D:\Ollama\models\manifests\registry.ollama.ai\library\nomic-embed-text\v1.5
PDF_DIR: D:\MyProjects\AI Project\Rag_local_Ollama\pdf
CHROMA_DIR: D:\MyProjects\AI Project\Rag_local_Ollama


In [3]:
# Cell 3: Verify Ollama connection and models

client = ollama.Client(host=OLLAMA_HOST)





def _extract_model_names_from_api(models_resp):

    names = []



    if isinstance(models_resp, dict):

        raw_models = models_resp.get("models", [])

    elif hasattr(models_resp, "models"):

        raw_models = getattr(models_resp, "models") or []

    else:

        raw_models = []



    for item in raw_models:

        if isinstance(item, dict):

            name = item.get("name") or item.get("model")

        else:

            name = getattr(item, "name", None) or getattr(item, "model", None)



        if isinstance(name, str) and name.strip():

            names.append(name.strip())



    return sorted(set(names))





def _extract_model_names_from_manifests(models_dir: Path):

    names = []

    library_dir = models_dir / "manifests" / "registry.ollama.ai" / "library"



    if not library_dir.exists():

        return names



    for family_dir in library_dir.iterdir():

        if not family_dir.is_dir():

            continue

        for tag_file in family_dir.iterdir():

            if tag_file.is_file():

                names.append(f"{family_dir.name}:{tag_file.name}")



    return sorted(set(names))





try:

    models_resp = client.list()



    api_model_names = _extract_model_names_from_api(models_resp)

    manifest_model_names = _extract_model_names_from_manifests(OLLAMA_MODELS_DIR)



    if api_model_names:

        model_names = api_model_names

        source = "Ollama API"

    else:

        model_names = manifest_model_names

        source = "local manifest fallback"



    print("Ollama is reachable.")

    print(f"Detected models from: {source}")



    if not model_names:

        print("No models detected.")

    else:

        print("Available models:")

        for m in model_names:

            print(" -", m)



    if CHAT_MODEL not in model_names:

        print(f"\nWARNING: {CHAT_MODEL} not found. Pull it using: ollama pull {CHAT_MODEL}")

    if EMBED_MODEL not in model_names:

        print(f"WARNING: {EMBED_MODEL} not found. Pull it using: ollama pull {EMBED_MODEL}")



except Exception as e:

    print("Could not connect to Ollama server.")

    print("Start Ollama and make sure it is running on http://localhost:11434")

    print("Error:", e)

Ollama is reachable.
Detected models from: Ollama API
Available models:
 - mistral:7b
 - nomic-embed-text:v1.5


In [4]:
# Cell 4: Read PDFs and create character chunks

def extract_text_from_pdf(pdf_path: Path) -> str:

    reader = PdfReader(str(pdf_path))

    pages_text = []

    for page in reader.pages:

        page_text = page.extract_text() or ""

        pages_text.append(page_text)

    return "\n".join(pages_text)





def chunk_text(text: str, chunk_size: int = 1000, overlap: int = 150) -> List[str]:

    text = " ".join(text.split())

    if not text:

        return []



    chunks = []

    start = 0

    step = max(1, chunk_size - overlap)



    while start < len(text):

        end = start + chunk_size

        chunk = text[start:end]

        if chunk.strip():

            chunks.append(chunk)

        start += step



    return chunks





if not PDF_DIR.exists():

    raise FileNotFoundError(f"PDF directory not found: {PDF_DIR.resolve()}")



# Recursive search so all PDFs inside pdf/ are included

pdf_files = sorted(PDF_DIR.rglob("*.pdf"))

if not pdf_files:

    raise FileNotFoundError(f"No PDF files found in: {PDF_DIR.resolve()}")



documents = []

for pdf in tqdm(pdf_files, desc="Reading PDFs"):

    raw_text = extract_text_from_pdf(pdf)

    chunks = chunk_text(raw_text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP)

    for i, ch in enumerate(chunks):

        # Include parent folder in the id to avoid collisions

        safe_parent = str(pdf.parent).replace("\\", "_").replace(":", "")

        documents.append(

            {

                "id": f"{safe_parent}_{pdf.stem}_{i}",

                "text": ch,

                "source": str(pdf.relative_to(PDF_DIR)),

                "chunk_index": i,

            }

        )



print(f"PDF files: {len(pdf_files)}")

print(f"Total chunks: {len(documents)}")

print("Sample chunk preview:")

print(documents[0]["text"][:500] if documents else "No chunks created")

Reading PDFs:   0%|          | 0/41 [00:00<?, ?it/s]

Reading PDFs:  37%|███▋      | 15/41 [00:17<00:16,  1.61it/s]Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)
Ignoring wrong pointing object 20 0 (offset 0)
Ignoring wrong pointing object 101 0 (offset 0)
Ignoring wrong pointing object 202 0 (offset 0)
Ignoring wrong pointing object 582 0 (offset 0)
Ignoring wrong pointing object 601 0 (offset 0)
Ignoring wrong pointing object 603 0 (offset 0)
Reading PDFs: 100%|██████████| 41/41 [01:04<00:00,  1.57s/it]

PDF files: 41
Total chunks: 5820
Sample chunk preview:
The 7th Open Society Conference (OSC) 2025 Faculty of Law, Social and Political Science Universitas Terbuka DOI: 10.33830/osc.v3i1.7048 / ISSN: 3032-2227 39 Copyright © 2025, Dwi Rahmawati, Deviani Setyorini. Self-Concept of Makeup Enthusiast Female Students in a Beauty Brand Community: Pinkflash Dwi Rahmawati, Deviani Setyorini Communication Department, Sultan Ageng Tirtayasa University e-mail: deviani.setyorini@untirta.ac.id Abstract This study explores the self -concept of female university s


In [5]:
# Cell 5: Create ChromaDB collection and store embeddings

import os

from chromadb.api.client import SharedSystemClient



# Remove stale telemetry override values from older runs in this kernel

os.environ.pop("CHROMA_PRODUCT_TELEMETRY_IMPL", None)

os.environ.pop("CHROMA_TELEMETRY_IMPL", None)

os.environ["ANONYMIZED_TELEMETRY"] = "False"



# Ensure a clean Chroma singleton state in this notebook kernel

SharedSystemClient.clear_system_cache()



chroma_client = chromadb.PersistentClient(

    path=str(CHROMA_DIR),

    settings=Settings(anonymized_telemetry=False),

)



# Recreate collection for a clean run

try:

    chroma_client.delete_collection(name=COLLECTION_NAME)

except Exception:

    pass



collection = chroma_client.create_collection(name=COLLECTION_NAME)



batch_size = 32

for i in tqdm(range(0, len(documents), batch_size), desc="Embedding + storing"):

    batch = documents[i : i + batch_size]



    texts = [d["text"] for d in batch]

    ids = [d["id"] for d in batch]

    metadatas = [

        {"source": d["source"], "chunk_index": d["chunk_index"]}

        for d in batch

    ]



    embed_resp = client.embed(model=EMBED_MODEL, input=texts)

    embeddings = embed_resp["embeddings"]



    collection.add(

        ids=ids,

        embeddings=embeddings,

        documents=texts,

        metadatas=metadatas,

    )



print("Indexing complete.")

print("Total vectors in collection:", collection.count())

Embedding + storing: 100%|██████████| 182/182 [25:09<00:00,  8.29s/it]

Indexing complete.
Total vectors in collection: 5820


In [6]:
# Cell 6: Retrieval + generation functions

def retrieve_context(query: str, top_k: int = 4):

    q_embed = client.embed(model=EMBED_MODEL, input=[query])["embeddings"][0]

    result = collection.query(

        query_embeddings=[q_embed],

        n_results=top_k,

        include=["documents", "metadatas", "distances"],

    )



    docs = result["documents"][0]

    metas = result["metadatas"][0]

    dists = result["distances"][0]

    return docs, metas, dists





def ask_rag(query: str, top_k: int = 4) -> Dict:

    docs, metas, dists = retrieve_context(query, top_k=top_k)



    context_blocks = []

    for i, (doc, meta, dist) in enumerate(zip(docs, metas, dists), start=1):

        context_blocks.append(

            f"[Chunk {i}] source={meta.get('source')} chunk={meta.get('chunk_index')} distance={dist:.4f}\n{doc}"

        )

    context_text = "\n\n".join(context_blocks)



    prompt = f"""

You are a helpful, precise, and structured assistant. Answer questions strictly using only the information provided in the given context (PDF content).

Do not use prior knowledge, assumptions, or external information beyond the context.

If the answer is found in the context, respond in a clear and well-organized format using:

1. Paragraph Explanation:
Provide a concise and clear explanation of the answer.

2. Key Points:
- Present important details as bullet points.
- Keep them short and relevant.

3. Table (if applicable):
Use a table when comparing items, listing structured data, or summarizing information.

Example:
| Item | Description |
|------|------------|
| A    | Details from context |
| B    | Details from context |

If the answer is only partially available, provide only the available information and do not fill gaps.

If the answer is NOT found in the context, respond exactly with:
"I could not find that in the provided PDFs."

Ensure the answer is:
- Accurate
- Context-based
- Well-structured
- Easy to understand



Question:

{query}



Context:

{context_text}

""".strip()



    response = client.chat(

        model=CHAT_MODEL,

        messages=[

            {"role": "system", "content": "You are an accurate RAG assistant."},

            {"role": "user", "content": prompt},

        ],

        options={"temperature": 0.2},

    )



    answer = response["message"]["content"]

    return {

        "question": query,

        "answer": answer,

        "retrieved_docs": docs,

        "retrieved_meta": metas,

        "retrieved_distances": dists,

    }

In [7]:
# Cell 7: Ask questions

question = "explain atleast 100 words >  what is the beauticulture"

result = ask_rag(question, top_k=4)



print("Question:", result["question"])

print("\nAnswer:\n", result["answer"])



print("\nRetrieved sources:")

for i, (meta, dist) in enumerate(zip(result["retrieved_meta"], result["retrieved_distances"]), start=1):

    print(f"{i}. {meta.get('source')} | chunk {meta.get('chunk_index')} | distance={dist:.4f}")



# Try your own question:

# result = ask_rag("Your question here", top_k=4)

# print(result["answer"])


Question: explain atleast 100 words >  what is the beauticulture

Answer:
  Paragraph Explanation:
The provided context does not explicitly define the term "beauticulture." However, it discusses the evolution and various aspects of the beauty industry throughout history. The beauty industry encompasses practices such as makeup, haircare, skincare, and grooming, which are considered components of a person's outward appearance. These practices serve as a means of self-expression and can be influenced by factors like physical attributes, facial features, clothing, cosmetics, and other grooming practices. As living standards rise, individuals become increasingly preoccupied with their appearance, leading to an increased interest in the beauty industry.

Key Points:
- The beauty industry includes makeup, haircare, skincare, and grooming practices.
- These practices serve as a means of self-expression.
- Interest in the beauty industry is influenced by factors like physical attributes, facia

In [13]:
# Cell 8: Translate latest RAG answer to Sinhala using Google Translate API (.env)

import os
import json
import html
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import Request, urlopen
from urllib.error import HTTPError

def load_google_translate_api_key() -> str:
    env_candidates = []

    # Prefer project root path configured earlier in Cell 2
    if "CHROMA_DIR" in globals():
        env_candidates.append(Path(CHROMA_DIR) / ".env")

    # Fallback to current working directory
    env_candidates.append(Path.cwd() / ".env")

    env_file = next((p for p in env_candidates if p.exists()), None)
    if env_file is not None:
        for raw_line in env_file.read_text(encoding="utf-8").splitlines():
            line = raw_line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))

    api_key = os.getenv("GOOGLE_TRANSLATE_API_KEY")
    if not api_key:
        searched = " | ".join(str(p) for p in env_candidates)
        raise ValueError(
            "GOOGLE_TRANSLATE_API_KEY not found. "
            f"Checked .env in: {searched}"
        )

    return api_key

def translate_to_sinhala_google(text: str, api_key: str) -> str:
    if not text or not text.strip():
        return ""

    payload = urlencode(
        {
            "q": text,
            "target": "si",
            "format": "text",
            "key": api_key,
        }
    ).encode("utf-8")

    request = Request(
        "https://translation.googleapis.com/language/translate/v2",
        data=payload,
        headers={"Content-Type": "application/x-www-form-urlencoded"},
        method="POST",
    )

    try:
        with urlopen(request, timeout=60) as response:
            response_data = json.loads(response.read().decode("utf-8"))
    except HTTPError as e:
        details = e.read().decode("utf-8", errors="ignore")
        raise RuntimeError(f"Google Translate API error ({e.code}): {details}") from e

    translated = response_data["data"]["translations"][0]["translatedText"]
    return html.unescape(translated)

try:
    if "result" not in globals() or "answer" not in result:
        raise NameError("result not found")

    api_key = load_google_translate_api_key()
    english_answer = result["answer"]
    sinhala_answer = translate_to_sinhala_google(english_answer, api_key)

    print("Sinhala Translation:\n")
    print(sinhala_answer)
except NameError:
    print("result not found. Run Cell 7 first.")
except Exception as e:
    print("Translation failed:", e)

Sinhala Translation:

ඡේද පැහැදිලි කිරීම:
සපයා ඇති සන්දර්භය "රූපලාවන්‍ය වගාව" යන පදය පැහැදිලිව නිර්වචනය නොකරයි. කෙසේ වෙතත්, එය ඉතිහාසය පුරා රූපලාවන්‍ය කර්මාන්තයේ පරිණාමය සහ විවිධ අංශ සාකච්ඡා කරයි. රූපලාවන්‍ය කර්මාන්තය පුද්ගලයෙකුගේ බාහිර පෙනුමේ සංරචක ලෙස සැලකෙන වේශ නිරූපණය, කොණ්ඩා රැකවරණය, සම රැකවරණය සහ අලංකාර කිරීම වැනි පිළිවෙත් ඇතුළත් වේ. මෙම පිළිවෙත් ස්වයං ප්‍රකාශනයේ මාධ්‍යයක් ලෙස සේවය කරන අතර භෞතික ගුණාංග, මුහුණේ ලක්ෂණ, ඇඳුම් පැළඳුම්, ආලේපන සහ වෙනත් අලංකාර කිරීමේ පිළිවෙත් වැනි සාධක මගින් බලපෑම් කළ හැකිය. ජීවන තත්වයන් ඉහළ යන විට, පුද්ගලයන් තම පෙනුම ගැන වැඩි වැඩියෙන් උනන්දු වන අතර, එමඟින් රූපලාවන්‍ය කර්මාන්තය කෙරෙහි වැඩි උනන්දුවක් ඇති වේ.

ප්‍රධාන කරුණු:
- රූපලාවන්‍ය කර්මාන්තයට වේශ නිරූපණය, කොණ්ඩා රැකවරණය, සම රැකවරණය සහ අලංකාර කිරීමේ පිළිවෙත් ඇතුළත් වේ.
- මෙම පිළිවෙත් ස්වයං ප්‍රකාශනයේ මාධ්‍යයක් ලෙස සේවය කරයි.
- රූපලාවන්‍ය කර්මාන්තය කෙරෙහි ඇති උනන්දුව භෞතික ගුණාංග, මුහුණේ ලක්ෂණ, ඇඳුම් පැළඳුම්, ආලේපන සහ වෙනත් අලංකාර කිරීමේ පිළිවෙත් වැනි සාධක මගින් බලපායි.
- ජීවන තත්වයන් ඉහළ යන විට, පුද්